# Computer Vision Project (Semester VIII)

## Task: Multi-Label Image Classification (Pascal VOC)

### Objective
Train and compare **3 pre-trained CNN backbones** for multi-label classification, using regularization techniques (**Dropout**, **Early Stopping**, **Weight Decay**) and analyze their impact.

### Framework
**PyTorch** + TorchVision

### Dataset
**Pascal VOC 2007 + 2012 (trainval)**
- Multi-label target = which of the 20 VOC object classes appear in each image.
- Dataset size is well above the minimum requirement (≥ 5000 images).

---

> Notes for reproducibility:
> - Run cells top-to-bottom.
> - If you have a GPU, training will be much faster. The notebook auto-detects CUDA.


In [ ]:
# If you're running in a fresh environment, install dependencies.
# In a configured environment, you can skip this cell.

%pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip -q install tqdm scikit-learn matplotlib pandas pillow


In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.datasets import VOCDetection

from tqdm.auto import tqdm

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import average_precision_score, precision_recall_fscore_support


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Torch:", torch.__version__)


In [ ]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]
CLASS_TO_IDX = {c: i for i, c in enumerate(VOC_CLASSES)}
NUM_CLASSES = len(VOC_CLASSES)


def voc_annotation_to_multihot(annotation: dict) -> torch.Tensor:
    # VOC XML is already parsed into dict by VOCDetection
    # annotation structure: {'annotation': {'object': [..] or {...}, ...}}
    objects = annotation.get("annotation", {}).get("object", [])
    if isinstance(objects, dict):
        objects = [objects]

    y = torch.zeros(NUM_CLASSES, dtype=torch.float32)
    for obj in objects:
        name = obj.get("name")
        if name in CLASS_TO_IDX:
            y[CLASS_TO_IDX[name]] = 1.0
    return y


class VOCMultiLabel(Dataset):
    def __init__(self, root: str, year: str, image_set: str, transform=None, download: bool = True):
        self.ds = VOCDetection(root=root, year=year, image_set=image_set, download=download)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.ds)

    def __getitem__(self, idx: int):
        img, ann = self.ds[idx]
        y = voc_annotation_to_multihot(ann)
        if self.transform is not None:
            img = self.transform(img)
        return img, y


In [ ]:
@dataclass
class TrainConfig:
    data_root: str = "./data"
    img_size: int = 224
    batch_size: int = 32
    num_workers: int = 2

    lr: float = 3e-4
    weight_decay: float = 1e-4  # L2 regularization
    epochs: int = 10

    # Regularization toggles
    use_dropout: bool = True
    dropout_p: float = 0.3

    use_early_stopping: bool = True
    early_stopping_patience: int = 3

    use_weight_decay: bool = True

    # Train/val split
    val_frac: float = 0.15


cfg = TrainConfig()
cfg

In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Download VOC 2007+2012 trainval (shared files on disk; different transforms per split)
voc07_train = VOCMultiLabel(root=cfg.data_root, year="2007", image_set="trainval", transform=train_tfms, download=True)
voc12_train = VOCMultiLabel(root=cfg.data_root, year="2012", image_set="trainval", transform=train_tfms, download=True)

voc07_val = VOCMultiLabel(root=cfg.data_root, year="2007", image_set="trainval", transform=val_tfms, download=False)
voc12_val = VOCMultiLabel(root=cfg.data_root, year="2012", image_set="trainval", transform=val_tfms, download=False)

full_train = torch.utils.data.ConcatDataset([voc07_train, voc12_train])
full_val = torch.utils.data.ConcatDataset([voc07_val, voc12_val])

n = len(full_train)
idxs = np.arange(n)
np.random.shuffle(idxs)
val_n = int(cfg.val_frac * n)
val_idxs = idxs[:val_n]
train_idxs = idxs[val_n:]

train_ds = torch.utils.data.Subset(full_train, train_idxs.tolist())
val_ds = torch.utils.data.Subset(full_val, val_idxs.tolist())

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

print("Total images:", n)
print("Train images:", len(train_ds))
print("Val images:", len(val_ds))


In [ ]:
def build_backbone(name: str, num_classes: int, use_dropout: bool, dropout_p: float) -> nn.Module:
    name = name.lower()

    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_f = m.fc.in_features
        head = [nn.Linear(in_f, 512), nn.ReLU(inplace=True)]
        if use_dropout:
            head.append(nn.Dropout(dropout_p))
        head.append(nn.Linear(512, num_classes))
        m.fc = nn.Sequential(*head)
        return m

    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_f = m.classifier[1].in_features
        head = [nn.Linear(in_f, 512), nn.SiLU(inplace=True)]
        if use_dropout:
            head.append(nn.Dropout(dropout_p))
        head.append(nn.Linear(512, num_classes))
        m.classifier = nn.Sequential(*head)
        return m

    if name == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        in_f = m.classifier[3].in_features
        head = [nn.Linear(in_f, 512), nn.Hardswish(inplace=True)]
        if use_dropout:
            head.append(nn.Dropout(dropout_p))
        head.append(nn.Linear(512, num_classes))
        m.classifier = nn.Sequential(*head)
        return m

    raise ValueError(f"Unknown backbone: {name}")


def sigmoid_threshold_metrics(y_true: np.ndarray, y_prob: np.ndarray, thr: float = 0.5) -> Dict[str, float]:
    y_pred = (y_prob >= thr).astype(np.int32)

    # micro precision/recall/f1 across all labels
    p, r, f1, _ = precision_recall_fscore_support(y_true.ravel(), y_pred.ravel(), average="binary", zero_division=0)

    # subset accuracy (exact match across all 20 labels) is strict but common for multi-label
    subset_acc = (y_pred == y_true).all(axis=1).mean()

    return {
        "precision_micro": float(p),
        "recall_micro": float(r),
        "f1_micro": float(f1),
        "subset_accuracy": float(subset_acc),
    }


def multilabel_map(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    # mean average precision over classes
    ap_per_class = []
    for k in range(y_true.shape[1]):
        # average_precision_score needs both positive and negative samples; skip degenerate classes in val
        if len(np.unique(y_true[:, k])) < 2:
            continue
        ap_per_class.append(average_precision_score(y_true[:, k], y_prob[:, k]))
    return float(np.mean(ap_per_class)) if ap_per_class else float("nan")


In [ ]:
@torch.no_grad()
def run_eval(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    all_true = []
    all_prob = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        prob = torch.sigmoid(logits)
        all_true.append(y.detach().cpu().numpy())
        all_prob.append(prob.detach().cpu().numpy())

    y_true = np.concatenate(all_true, axis=0)
    y_prob = np.concatenate(all_prob, axis=0)

    out = sigmoid_threshold_metrics(y_true, y_prob, thr=0.5)
    out["mAP"] = multilabel_map(y_true, y_prob)
    return out


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    losses = []

    for x, y in tqdm(loader, desc="train", leave=False):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))


def fit_model(
    backbone: str,
    base_cfg: TrainConfig,
) -> Tuple[nn.Module, pd.DataFrame]:
    model = build_backbone(backbone, NUM_CLASSES, base_cfg.use_dropout, base_cfg.dropout_p).to(device)

    wd = base_cfg.weight_decay if base_cfg.use_weight_decay else 0.0
    optimizer = torch.optim.AdamW(model.parameters(), lr=base_cfg.lr, weight_decay=wd)

    best_val = -np.inf
    best_state = None
    patience_left = base_cfg.early_stopping_patience

    rows = []

    for epoch in range(1, base_cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_metrics = run_eval(model, val_loader)

        row = {"epoch": epoch, "train_loss": train_loss, **val_metrics}
        rows.append(row)
        print(row)

        score = val_metrics.get("mAP", float("nan"))
        if np.isnan(score):
            score = val_metrics.get("f1_micro", 0.0)

        improved = score > best_val
        if improved:
            best_val = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = base_cfg.early_stopping_patience
        else:
            if base_cfg.use_early_stopping:
                patience_left -= 1
                if patience_left <= 0:
                    print(f"Early stopping at epoch {epoch} (best_score={best_val:.4f}).")
                    break

    hist = pd.DataFrame(rows)

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, hist


In [ ]:
def run_experiments() -> pd.DataFrame:
    backbones = ["resnet50", "efficientnet_b0", "mobilenet_v3_large"]

    # Required comparison: with vs without each regularizer.
    # We keep the other two ON while toggling one OFF.
    settings = [
        ("all_on", dict(use_dropout=True, use_early_stopping=True, use_weight_decay=True)),
        ("no_dropout", dict(use_dropout=False, use_early_stopping=True, use_weight_decay=True)),
        ("no_early_stopping", dict(use_dropout=True, use_early_stopping=False, use_weight_decay=True)),
        ("no_weight_decay", dict(use_dropout=True, use_early_stopping=True, use_weight_decay=False)),
    ]

    summary_rows = []

    for backbone in backbones:
        for tag, overrides in settings:
            exp_cfg = TrainConfig(**{**cfg.__dict__, **overrides})
            print("\n" + "=" * 90)
            print(f"Backbone={backbone} | Setting={tag} | cfg={overrides}")
            print("=" * 90)

            model, hist = fit_model(backbone, exp_cfg)
            final_metrics = run_eval(model, val_loader)

            summary_rows.append({
                "backbone": backbone,
                "setting": tag,
                **overrides,
                **{f"final_{k}": v for k, v in final_metrics.items()},
                "epochs_ran": int(hist["epoch"].max()) if len(hist) else 0,
            })

    return pd.DataFrame(summary_rows)


# WARNING: This can take a while on CPU. Consider lowering cfg.epochs or cfg.batch_size.
results_df = run_experiments()
results_df

In [ ]:
# Comparative analysis helpers

def plot_metric(results: pd.DataFrame, metric: str):
    pivot = results.pivot(index="backbone", columns="setting", values=metric)
    ax = pivot.plot(kind="bar", figsize=(12, 5))
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


display(results_df.sort_values(["backbone", "setting"]))

for m in ["final_mAP", "final_f1_micro", "final_precision_micro", "final_recall_micro", "final_subset_accuracy"]:
    if m in results_df.columns:
        plot_metric(results_df, m)


## Discussion / Conclusion (write-up)

Use the plots and `results_df` table to discuss:

- **Backbone comparison**: which pre-trained model performs best (mAP / F1) and why (capacity, inductive biases, optimization stability, etc.).
- **Dropout effect**: did it reduce overfitting? Did it hurt if the model was already regularized?
- **Early stopping effect**: did it prevent over-training? Did it stop too early when learning was slow?
- **Weight decay effect**: did it improve generalization? Did too much regularization reduce performance?

### Suggested points to include
- Overfitting signs: training loss down while validation mAP/F1 stagnates or drops.
- Effect of dataset imbalance: some classes appear rarely → AP instability.
- Thresholding: fixed 0.5 may not be optimal; mAP is threshold-free and usually preferred.

---

## Submission checklist
- Notebook contains: title/description, dataset details, preprocessing/augmentation, 3 model architectures, training details, results (Accuracy/Precision/Recall/F1 + mAP), comparative analysis, conclusion, full code.
